In [ ]:
# -*- coding: utf-8 -*-
"""Copy of MedConflict_Bench_v2.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RA3NzGv00AKDyf3sFePcIO1MAO3aYY0Q
"""

#!/usr/bin/env python3
# =============================================================================
# MedConflict-Bench v2 Builder — Colab Notebook
# =============================================================================
#
# INSTRUCTIONS:
#   1. Create a new Colab notebook: MedConflict_Bench_v2.ipynb
#   2. Copy each "# ── CELL N" block into a separate Colab cell
#   3. Run cells in order. If runtime disconnects, just re-run —
#      each cell checks for saved checkpoints and skips if already done.
#
# All intermediate results are saved to Google Drive under:
#   NIPS_2026/MedConflict-Bench/checkpoints/
#
# Author: Ala Sai Teja, IIIT Sricity
# Project: GenMedAssist — NeurIPS 2026
# =============================================================================


# ── CELL 1: Mount Drive & Setup ─────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import json
import hashlib
import os
import pickle
import warnings
import gc
from pathlib import Path
from datetime import timedelta
from collections import defaultdict

warnings.filterwarnings("ignore")

# === PATHS (edit only this block) ===
BASE_DIR    = Path("/content/drive/MyDrive/NIPS_2026")
DATA_DIR    = BASE_DIR / "Dataset" / "mimic-iv-hosp"
OUTPUT_DIR  = BASE_DIR / "MedConflict-Bench"
CKPT_DIR    = OUTPUT_DIR / "checkpoints"

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# === CONFIG ===
TARGET_PER_STRATUM = 500
TARGET_SAFE = 500
LAB_WINDOW_HOURS = 48
SEED = 42
np.random.seed(SEED)

print("✅ Drive mounted, directories ready")
print(f"   Data:        {DATA_DIR}")
print(f"   Output:      {OUTPUT_DIR}")
print(f"   Checkpoints: {CKPT_DIR}")

# Quick sanity check
assert (DATA_DIR / "prescriptions.csv.gz").exists(), f"❌ prescriptions.csv.gz not found in {DATA_DIR}"
print("✅ MIMIC-IV files found")

In [ ]:
# ── CELL 2: Helper Functions ────────────────────────────────────────────────

def ckpt_exists(name):
    """Check if a checkpoint file exists on Drive."""
    return (CKPT_DIR / f"{name}.pkl").exists()

def save_ckpt(name, data):
    """Save checkpoint to Drive (survives runtime disconnect)."""
    path = CKPT_DIR / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"   💾 Checkpoint saved: {name} ({path.stat().st_size / 1024 / 1024:.1f} MB)")

def load_ckpt(name):
    """Load checkpoint from Drive."""
    path = CKPT_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
    print(f"   📂 Checkpoint loaded: {name}")
    return data

def compute_egfr_ckdepi2021(creatinine, age, is_female):
    """CKD-EPI 2021 race-free eGFR from serum creatinine."""
    if pd.isna(creatinine) or pd.isna(age) or creatinine <= 0 or age <= 0:
        return np.nan
    kappa = 0.7 if is_female else 0.9
    alpha = -0.241 if is_female else -0.302
    scr_ratio = creatinine / kappa
    egfr = 142 * (min(scr_ratio, 1.0) ** alpha) * (max(scr_ratio, 1.0) ** (-1.200)) * (0.9938 ** age)
    if is_female:
        egfr *= 1.012
    return round(egfr, 2)

def generate_example_id(stratum, rule_id, subject_id, hadm_id):
    """Deterministic unique ID for each example."""
    raw = f"{stratum}_{rule_id}_{subject_id}_{hadm_id}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]

print("✅ Helper functions defined")

import pandas as pd

In [ ]:
# ── CELL 3: Load MIMIC-IV Tables (with checkpoint) ──────────────────────────

if ckpt_exists("mimic_tables"):
    tables = load_ckpt("mimic_tables")
    prescriptions = tables["prescriptions"]
    labevents     = tables["labevents"]
    patients      = tables["patients"]
    diagnoses     = tables["diagnoses"]
else:
    print("Loading MIMIC-IV tables (this takes 3-5 min, saves to Drive after)...")

    print("  prescriptions.csv.gz ...", end=" ")
    prescriptions = pd.read_csv(
        DATA_DIR / "prescriptions.csv.gz", compression="gzip",
        usecols=["subject_id", "hadm_id", "drug", "dose_val_rx", "dose_unit_rx",
                 "route", "starttime"],
        dtype={
            "subject_id": "Int32",
            "hadm_id": "Int32",
            "drug": "category",
            "dose_val_rx": "string",
            "dose_unit_rx": "category",
            "route": "category"
        }
    )
    prescriptions["starttime"] = pd.to_datetime(prescriptions["starttime"], errors="coerce")
    prescriptions = prescriptions.dropna(subset=["hadm_id", "drug"])
    prescriptions["hadm_id"] = prescriptions["hadm_id"].astype("Int32") # Ensure consistent Int32 after dropna
    print(f"({len(prescriptions):,} rows)")
    prescriptions.info(memory_usage='deep')

    print("  labevents.csv.gz ...", end=" ")
    labevents = pd.read_csv(
        DATA_DIR / "labevents.csv.gz", compression="gzip",
        usecols=["subject_id", "hadm_id", "itemid", "valuenum", "valueuom", "charttime"],
        dtype={
            "subject_id": "Int32",
            "hadm_id": "Int32",
            "itemid": "Int32",
            "valuenum": float,
            "valueuom": "category"
        }
    )
    labevents["charttime"] = pd.to_datetime(labevents["charttime"], errors="coerce")
    labevents = labevents.dropna(subset=["hadm_id", "valuenum"])
    labevents["hadm_id"] = labevents["hadm_id"].astype("Int32") # Ensure consistent Int32 after dropna
    print(f"({len(labevents):,} rows)")
    labevents.info(memory_usage='deep')

    print("  patients.csv.gz ...", end=" ")
    patients = pd.read_csv(
        DATA_DIR / "patients.csv.gz", compression="gzip",
        usecols=["subject_id", "gender", "anchor_age"],
        dtype={
            "subject_id": "Int32",
            "gender": "category",
            "anchor_age": "Int8"
        }
    )
    print(f"({len(patients):,} rows)")
    patients.info(memory_usage='deep')

    print("  diagnoses_icd.csv.gz ...", end=" ")
    diagnoses = pd.read_csv(
        DATA_DIR / "diagnoses_icd.csv.gz", compression="gzip",
        usecols=["subject_id", "hadm_id", "icd_code", "icd_version"],
        dtype={
            "subject_id": "Int32",
            "hadm_id": "Int32",
            "icd_code": "category",
            "icd_version": "Int8"
        }
    )
    print(f"({len(diagnoses):,} rows)")
    diagnoses.info(memory_usage='deep')

    # Merge patient demographics into prescriptions (needed for eGFR)
    print("  Merging patient demographics into prescriptions...")
    prescriptions = prescriptions.merge(patients, on="subject_id", how="left")
    prescriptions.info(memory_usage='deep')

    # Save checkpoint
    save_ckpt("mimic_tables", {
        "prescriptions": prescriptions,
        "labevents": labevents,
        "patients": patients,
        "diagnoses": diagnoses,
    })

print(f"\n📊 Loaded:")
print(f"   Prescriptions: {len(prescriptions):,}")
print(f"   Lab events:    {len(labevents):,}")
print(f"   Patients:      {len(patients):,}")
print(f"   Diagnoses:     {len(diagnoses):,}")

In [ ]:
# ── CELL 4: Define All Rules ────────────────────────────────────────────────

# ---------- STRATUM 1: Categorical (drug-allergy, drug-disease) ----------
CATEGORICAL_RULES = [
    {
        "rule_id": "CAT-001",
        "drug_pattern": r"(?i)\b(amoxicillin|ampicillin|piperacillin|nafcillin|oxacillin|penicillin)\b",
        "drug_class": "Penicillin-class antibiotics",
        "conflict_condition": "Penicillin Allergy",
        "icd10_codes": ["Z88.0"], "icd9_codes": ["V14.0"],
        "conflict_type": "drug_allergy", "severity": "major",
        "action": "contraindicated",
        "explanation": "Patient has documented penicillin allergy; prescribing penicillin-class antibiotic risks anaphylaxis."
    },
    {
        "rule_id": "CAT-002",
        "drug_pattern": r"(?i)\b(sulfamethoxazole|sulfasalazine|sulfadiazine|trimethoprim.sulfamethoxazole|bactrim|septra)\b",
        "drug_class": "Sulfonamide antibiotics",
        "conflict_condition": "Sulfonamide Allergy",
        "icd10_codes": ["Z88.2"], "icd9_codes": ["V14.2"],
        "conflict_type": "drug_allergy", "severity": "major",
        "action": "contraindicated",
        "explanation": "Patient has documented sulfonamide allergy; prescribing sulfa drug risks severe allergic reaction."
    },
    {
        "rule_id": "CAT-003",
        "drug_pattern": r"(?i)\b(cephalexin|cefazolin|ceftriaxone|cefepime|cefuroxime|cephalosporin)\b",
        "drug_class": "Cephalosporin antibiotics",
        "conflict_condition": "Penicillin Allergy (cross-reactivity)",
        "icd10_codes": ["Z88.0"], "icd9_codes": ["V14.0"],
        "conflict_type": "drug_allergy_cross_reactivity", "severity": "moderate",
        "action": "use_with_caution",
        "explanation": "Patient has penicillin allergy; cephalosporins have ~2-5% cross-reactivity risk."
    },
    {
        "rule_id": "CAT-004",
        "drug_pattern": r"(?i)\b(metformin)\b",
        "drug_class": "Biguanides",
        "conflict_condition": "Heart Failure (severe/decompensated)",
        "icd10_codes": ["I50.2","I50.3","I50.4","I50.21","I50.22","I50.23",
                         "I50.31","I50.32","I50.33","I50.41","I50.42","I50.43"],
        "icd9_codes": ["428.0","428.1","428.2","428.3","428.4"],
        "conflict_type": "drug_disease", "severity": "major",
        "action": "contraindicated",
        "explanation": "Metformin is contraindicated in decompensated heart failure due to lactic acidosis risk."
    },
    {
        "rule_id": "CAT-005",
        "drug_pattern": r"(?i)\b(ibuprofen|naproxen|ketorolac|indomethacin|diclofenac|meloxicam|celecoxib)\b",
        "drug_class": "NSAIDs",
        "conflict_condition": "Active GI Bleeding",
        "icd10_codes": ["K25.0","K25.2","K25.4","K25.6","K26.0","K26.2",
                         "K26.4","K26.6","K92.0","K92.1","K92.2"],
        "icd9_codes": ["531.0","531.2","531.4","532.0","578.0","578.1","578.9"],
        "conflict_type": "drug_disease", "severity": "major",
        "action": "contraindicated",
        "explanation": "NSAIDs contraindicated in active GI bleeding; inhibit platelet aggregation and worsen hemorrhage."
    },
    {
        "rule_id": "CAT-006",
        "drug_pattern": r"(?i)\b(methotrexate)\b",
        "drug_class": "Antimetabolites",
        "conflict_condition": "Pregnancy",
        "icd10_codes": ["Z33.1","O09.0","O09.1","O09.2","O09.3",
                         "O09.4","O09.5","O09.6","O09.7","O09.9"],
        "icd9_codes": ["V22.0","V22.1","V22.2"],
        "conflict_type": "drug_disease", "severity": "major",
        "action": "contraindicated",
        "explanation": "Methotrexate is a known teratogen; absolutely contraindicated in pregnancy."
    },
    {
        "rule_id": "CAT-007",
        "drug_pattern": r"(?i)\b(warfarin|coumadin)\b",
        "drug_class": "Vitamin K antagonists",
        "conflict_condition": "Pregnancy",
        "icd10_codes": ["Z33.1","O09.0","O09.1","O09.2","O09.3",
                         "O09.4","O09.5","O09.6","O09.7","O09.9"],
        "icd9_codes": ["V22.0","V22.1","V22.2"],
        "conflict_type": "drug_disease", "severity": "major",
        "action": "contraindicated",
        "explanation": "Warfarin is teratogenic (warfarin embryopathy); contraindicated in pregnancy."
    },
    {
        "rule_id": "CAT-008",
        "drug_pattern": r"(?i)\b(ciprofloxacin|levofloxacin|moxifloxacin)\b",
        "drug_class": "Fluoroquinolones",
        "conflict_condition": "Myasthenia Gravis",
        "icd10_codes": ["G70.0","G70.00","G70.01"],
        "icd9_codes": ["358.0","358.00","358.01"],
        "conflict_type": "drug_disease", "severity": "major",
        "action": "contraindicated",
        "explanation": "Fluoroquinolones may exacerbate muscle weakness in myasthenia gravis; risk of respiratory failure."
    },
    {
        "rule_id": "CAT-009",
        "drug_pattern": r"(?i)\b(aspirin)\b",
        "drug_class": "Salicylates",
        "conflict_condition": "Aspirin/Analgesic Allergy",
        "icd10_codes": ["Z88.6"], "icd9_codes": ["V14.6"],
        "conflict_type": "drug_allergy", "severity": "major",
        "action": "contraindicated",
        "explanation": "Patient has documented analgesic allergy; aspirin may cause bronchospasm or anaphylaxis."
    },
    {
        "rule_id": "CAT-010",
        "drug_pattern": r"(?i)\b(metoprolol|atenolol|propranolol|carvedilol|bisoprolol|nadolol|labetalol)\b",
        "drug_class": "Beta-blockers",
        "conflict_condition": "Severe Bradycardia / AV Block",
        "icd10_codes": ["I44.1","I44.2","I44.3","R00.1"],
        "icd9_codes": ["426.0","426.10","426.11","426.12","427.81"],
        "conflict_type": "drug_disease", "severity": "major",
        "action": "contraindicated",
        "explanation": "Beta-blockers contraindicated in severe bradycardia/high-degree AV block; risk of asystole."
    },
    {
        "rule_id": "CAT-011",
        "drug_pattern": r"(?i)\b(morphine|oxycodone|hydromorphone|fentanyl|codeine|hydrocodone|tramadol)\b",
        "drug_class": "Opioids",
        "conflict_condition": "Respiratory Failure / Severe COPD",
        "icd10_codes": ["J96.0","J96.1","J96.9","J44.1"],
        "icd9_codes": ["518.81","518.82","518.84","491.21"],
        "conflict_type": "drug_disease", "severity": "major",
        "action": "use_with_extreme_caution",
        "explanation": "Opioids risk fatal respiratory depression in patients with existing respiratory failure."
    },
    {
        "rule_id": "CAT-012",
        "drug_pattern": r"(?i)\b(glipizide|glyburide|glimepiride)\b",
        "drug_class": "Sulfonylureas",
        "conflict_condition": "Sulfonamide Allergy (cross-reactivity)",
        "icd10_codes": ["Z88.2"], "icd9_codes": ["V14.2"],
        "conflict_type": "drug_allergy_cross_reactivity", "severity": "moderate",
        "action": "use_with_caution",
        "explanation": "Sulfonylureas share sulfonamide moiety; potential cross-reactivity in sulfa-allergic patients."
    },
]

# ---------- STRATUM 2: Numerical-Threshold ----------
THRESHOLD_RULES = [
    {"rule_id":"NUM-001","drug_pattern":r"(?i)\b(metformin)\b","lab_name":"eGFR","lab_source":"computed","lab_itemid":50912,"operator":"<","threshold":30,"severity":"major","action":"contraindicated","condition":"Renal Impairment","explanation":"Metformin contraindicated when eGFR < 30; lactic acidosis risk."},
    {"rule_id":"NUM-002","drug_pattern":r"(?i)\b(metformin)\b","lab_name":"eGFR","lab_source":"computed","lab_itemid":50912,"operator":"<","threshold":45,"severity":"moderate","action":"dose_reduction","condition":"Renal Impairment","explanation":"Metformin requires dose reduction when eGFR 30-45."},
    {"rule_id":"NUM-003","drug_pattern":r"(?i)\b(gabapentin)\b","lab_name":"eGFR","lab_source":"computed","lab_itemid":50912,"operator":"<","threshold":60,"severity":"moderate","action":"dose_reduction","condition":"Renal Impairment","explanation":"Gabapentin dose must be adjusted when eGFR < 60."},
    {"rule_id":"NUM-004","drug_pattern":r"(?i)\b(dabigatran|pradaxa)\b","lab_name":"eGFR","lab_source":"computed","lab_itemid":50912,"operator":"<","threshold":30,"severity":"major","action":"contraindicated","condition":"Renal Impairment","explanation":"Dabigatran contraindicated when eGFR < 30; accumulation and bleeding risk."},
    {"rule_id":"NUM-005","drug_pattern":r"(?i)\b(rivaroxaban|xarelto)\b","lab_name":"eGFR","lab_source":"computed","lab_itemid":50912,"operator":"<","threshold":15,"severity":"major","action":"contraindicated","condition":"Renal Impairment","explanation":"Rivaroxaban contraindicated when eGFR < 15."},
    {"rule_id":"NUM-006","drug_pattern":r"(?i)\b(digoxin)\b","lab_name":"Potassium","lab_source":"direct","lab_itemid":50971,"operator":"<","threshold":3.5,"severity":"major","action":"risk_increased","condition":"Hypokalemia","explanation":"Digoxin toxicity risk markedly increased with K+ < 3.5 mEq/L."},
    {"rule_id":"NUM-007","drug_pattern":r"(?i)\b(spironolactone|aldactone)\b","lab_name":"Potassium","lab_source":"direct","lab_itemid":50971,"operator":">","threshold":5.0,"severity":"major","action":"contraindicated","condition":"Hyperkalemia","explanation":"Spironolactone contraindicated with K+ > 5.0; risk of fatal hyperkalemia."},
    {"rule_id":"NUM-008","drug_pattern":r"(?i)\b(lisinopril|enalapril|ramipril|captopril|benazepril|fosinopril|quinapril)\b","lab_name":"Potassium","lab_source":"direct","lab_itemid":50971,"operator":">","threshold":5.5,"severity":"major","action":"discontinue","condition":"Hyperkalemia","explanation":"ACE inhibitors must be discontinued with K+ > 5.5; life-threatening arrhythmia risk."},
    {"rule_id":"NUM-009","drug_pattern":r"(?i)\b(acetaminophen|tylenol|paracetamol)\b","lab_name":"ALT","lab_source":"direct","lab_itemid":50861,"operator":">","threshold":120,"severity":"major","action":"contraindicated","condition":"Hepatic Impairment","explanation":"Acetaminophen contraindicated when ALT > 120 U/L; hepatotoxicity risk."},
    {"rule_id":"NUM-010","drug_pattern":r"(?i)\b(atorvastatin|simvastatin|rosuvastatin|pravastatin|lovastatin)\b","lab_name":"ALT","lab_source":"direct","lab_itemid":50861,"operator":">","threshold":120,"severity":"major","action":"contraindicated","condition":"Hepatic Impairment","explanation":"Statins contraindicated when ALT > 3x ULN (~120); liver damage risk."},
    {"rule_id":"NUM-011","drug_pattern":r"(?i)\b(warfarin|coumadin)\b","lab_name":"INR","lab_source":"direct","lab_itemid":51237,"operator":">","threshold":4.0,"severity":"major","action":"hold_dose","condition":"Bleeding Risk","explanation":"Warfarin must be held when INR > 4.0; significant bleeding risk."},
    {"rule_id":"NUM-012","drug_pattern":r"(?i)\b(heparin)\b","lab_name":"Platelet Count","lab_source":"direct","lab_itemid":51265,"operator":"<","threshold":100000,"severity":"major","action":"contraindicated","condition":"Thrombocytopenia","explanation":"Heparin contraindicated with platelets < 100K; HIT and hemorrhage risk."},
    {"rule_id":"NUM-013","drug_pattern":r"(?i)\b(glipizide|glyburide|glimepiride)\b","lab_name":"Glucose","lab_source":"direct","lab_itemid":50931,"operator":"<","threshold":70,"severity":"moderate","action":"dose_reduction","condition":"Hypoglycemia Risk","explanation":"Sulfonylurea dose should be reduced with glucose < 70 mg/dL."},
    {"rule_id":"NUM-014","drug_pattern":r"(?i)\b(insulin)\b","lab_name":"Glucose","lab_source":"direct","lab_itemid":50931,"operator":"<","threshold":54,"severity":"major","action":"hold_dose","condition":"Hypoglycemia Risk","explanation":"Insulin must be held with glucose < 54; severe hypoglycemia (Level 2)."},
    {"rule_id":"NUM-015","drug_pattern":r"(?i)\b(methotrexate)\b","lab_name":"WBC","lab_source":"direct","lab_itemid":51300,"operator":"<","threshold":3000,"severity":"major","action":"hold_dose","condition":"Bone Marrow Suppression","explanation":"Methotrexate must be held with WBC < 3000; bone marrow suppression."},
    {"rule_id":"NUM-016","drug_pattern":r"(?i)\b(methotrexate)\b","lab_name":"eGFR","lab_source":"computed","lab_itemid":50912,"operator":"<","threshold":30,"severity":"major","action":"contraindicated","condition":"Renal Impairment","explanation":"Methotrexate contraindicated when eGFR < 30; accumulation toxicity."},
]

# ---------- STRATUM 3: Multi-Hop Causal ----------
MULTIHOP_RULES = [
    {"rule_id":"MH-001","drug_a_pattern":r"(?i)\b(warfarin|coumadin)\b","drug_b_pattern":r"(?i)\b(fluconazole|diflucan)\b","drug_a_name":"Warfarin","drug_b_name":"Fluconazole","mechanism":"Fluconazole inhibits CYP2C9 → decreased warfarin metabolism → elevated INR","lab_name":"INR","lab_itemid":51237,"operator":">","threshold":3.5,"severity":"major","action":"monitor_closely","condition":"Warfarin Toxicity via Drug Interaction","explanation":"Fluconazole inhibits CYP2C9, potentiating warfarin; INR > 3.5 indicates toxicity risk."},
    {"rule_id":"MH-002","drug_a_pattern":r"(?i)\b(warfarin|coumadin)\b","drug_b_pattern":r"(?i)\b(metronidazole|flagyl)\b","drug_a_name":"Warfarin","drug_b_name":"Metronidazole","mechanism":"Metronidazole inhibits CYP2C9 → decreased warfarin metabolism → elevated INR","lab_name":"INR","lab_itemid":51237,"operator":">","threshold":3.5,"severity":"major","action":"monitor_closely","condition":"Warfarin Toxicity via Drug Interaction","explanation":"Metronidazole inhibits warfarin metabolism via CYP2C9; INR > 3.5 with co-prescription indicates toxicity."},
    {"rule_id":"MH-003","drug_a_pattern":r"(?i)\b(warfarin|coumadin)\b","drug_b_pattern":r"(?i)\b(amiodarone|cordarone)\b","drug_a_name":"Warfarin","drug_b_name":"Amiodarone","mechanism":"Amiodarone inhibits CYP2C9/CYP3A4 → decreased warfarin metabolism → elevated INR","lab_name":"INR","lab_itemid":51237,"operator":">","threshold":3.5,"severity":"major","action":"dose_reduction","condition":"Warfarin Toxicity via Drug Interaction","explanation":"Amiodarone potentiates warfarin via CYP inhibition; INR > 3.5 signals over-anticoagulation."},
    {"rule_id":"MH-004","drug_a_pattern":r"(?i)\b(digoxin)\b","drug_b_pattern":r"(?i)\b(furosemide|lasix|bumetanide|bumex|torsemide)\b","drug_a_name":"Digoxin","drug_b_name":"Loop Diuretic","mechanism":"Loop diuretics cause K+ wasting → hypokalemia → increased digoxin toxicity","lab_name":"Potassium","lab_itemid":50971,"operator":"<","threshold":3.5,"severity":"major","action":"risk_increased","condition":"Digoxin Toxicity via Diuretic-Induced Hypokalemia","explanation":"Loop diuretics deplete potassium; hypokalemia (K+ < 3.5) potentiates digoxin toxicity."},
    {"rule_id":"MH-005","drug_a_pattern":r"(?i)\b(lisinopril|enalapril|ramipril|captopril|benazepril)\b","drug_b_pattern":r"(?i)\b(spironolactone|aldactone|eplerenone)\b","drug_a_name":"ACE Inhibitor","drug_b_name":"K-Sparing Diuretic","mechanism":"Both ACE-I and K-sparing diuretics increase K+ → hyperkalemia","lab_name":"Potassium","lab_itemid":50971,"operator":">","threshold":5.5,"severity":"major","action":"discontinue_one","condition":"Hyperkalemia via Dual K-Retaining Drugs","explanation":"ACE inhibitors + K-sparing diuretics both elevate K+; > 5.5 is life-threatening."},
    {"rule_id":"MH-006","drug_a_pattern":r"(?i)\b(methotrexate)\b","drug_b_pattern":r"(?i)\b(ibuprofen|naproxen|ketorolac|indomethacin|diclofenac|meloxicam)\b","drug_a_name":"Methotrexate","drug_b_name":"NSAID","mechanism":"NSAIDs reduce renal clearance of methotrexate → accumulation → bone marrow suppression","lab_name":"WBC","lab_itemid":51300,"operator":"<","threshold":3000,"severity":"major","action":"hold_methotrexate","condition":"Methotrexate Toxicity via NSAID-Reduced Clearance","explanation":"NSAIDs impair methotrexate renal clearance; WBC < 3000 with co-prescription indicates toxicity."},
    {"rule_id":"MH-007","drug_a_pattern":r"(?i)\b(lithium)\b","drug_b_pattern":r"(?i)\b(ibuprofen|naproxen|ketorolac|indomethacin|diclofenac|meloxicam)\b","drug_a_name":"Lithium","drug_b_name":"NSAID","mechanism":"NSAIDs reduce lithium renal clearance → elevated lithium levels → toxicity","lab_name":"Creatinine","lab_itemid":50912,"operator":">","threshold":1.5,"severity":"major","action":"monitor_lithium_levels","condition":"Lithium Toxicity Risk via NSAID Co-administration","explanation":"NSAIDs impair lithium clearance; rising creatinine (> 1.5) with co-prescription indicates accumulation risk."},
    {"rule_id":"MH-008","drug_a_pattern":r"(?i)\b(simvastatin|atorvastatin|lovastatin)\b","drug_b_pattern":r"(?i)\b(clarithromycin|erythromycin)\b","drug_a_name":"Statin (CYP3A4)","drug_b_name":"Macrolide Antibiotic","mechanism":"Macrolides inhibit CYP3A4 → statin accumulation → rhabdomyolysis → elevated ALT","lab_name":"ALT","lab_itemid":50861,"operator":">","threshold":120,"severity":"major","action":"hold_statin","condition":"Statin Toxicity via CYP3A4 Inhibition","explanation":"Macrolides inhibit CYP3A4 statin metabolism; ALT > 120 with co-prescription signals hepatotoxicity."},
]

print(f"✅ Rules defined: {len(CATEGORICAL_RULES)} categorical, {len(THRESHOLD_RULES)} threshold, {len(MULTIHOP_RULES)} multi-hop")


# ── MEMORY OPTIMIZATION HELPERS (47 GB RAM SAFE) ─────────────────────────────
# These helpers keep the same source data and the same output logic, but reduce
# RAM spikes by caching drug/lab filters, selecting only required columns before
# merges, using vectorized eGFR, and avoiding row-wise apply for exclusions.

RX_BASE_COLS = [
    "subject_id", "hadm_id", "drug", "dose_val_rx", "dose_unit_rx",
    "route", "starttime", "gender", "anchor_age"
]
LAB_BASE_COLS = ["subject_id", "hadm_id", "valuenum", "valueuom", "charttime"]

prescriptions["_drug_text"] = prescriptions["drug"].astype("string")
diagnoses["_icd_nodot"] = diagnoses["icd_code"].astype("string").str.replace(".", "", regex=False)

_DRUG_RX_CACHE = {}

def get_drug_rxs(pattern, cols=None):
    if pattern not in _DRUG_RX_CACHE:
        mask = prescriptions["_drug_text"].str.contains(pattern, regex=True, na=False)
        _DRUG_RX_CACHE[pattern] = prescriptions.loc[mask].copy()
    df = _DRUG_RX_CACHE[pattern]
    if cols is None:
        return df
    return df[[c for c in cols if c in df.columns]].copy()

def add_nearest_lab_within_window(rx_df, lab_df, start_col="starttime"):
    if rx_df.empty or lab_df.empty:
        return pd.DataFrame()
    merged = rx_df.merge(lab_df, on=["subject_id", "hadm_id"], how="inner", copy=False)
    if merged.empty:
        return merged
    merged["time_diff_h"] = ((merged["charttime"] - merged[start_col]).dt.total_seconds() / 3600).abs()
    merged = merged.loc[merged["time_diff_h"] <= LAB_WINDOW_HOURS]
    if merged.empty:
        return merged
    return merged.sort_values("time_diff_h").drop_duplicates(subset=["subject_id", "hadm_id"], keep="first")

def vectorized_egfr(creatinine, age, gender):
    scr = pd.to_numeric(creatinine, errors="coerce").astype("float64")
    age = pd.to_numeric(age, errors="coerce").astype("float64")
    is_female = gender.astype("string").str.upper().eq("F")
    kappa = np.where(is_female, 0.7, 0.9)
    alpha = np.where(is_female, -0.241, -0.302)
    ratio = scr / kappa
    egfr = 142 * (np.minimum(ratio, 1.0) ** alpha) * (np.maximum(ratio, 1.0) ** (-1.200)) * (0.9938 ** age)
    egfr = np.where(is_female, egfr * 1.012, egfr)
    egfr = np.where((scr > 0) & (age > 0), egfr, np.nan)
    return np.round(egfr, 2)

In [ ]:
# ── CELL 5: Build Stratum 1 — Categorical Conflicts ────────────────────────

if ckpt_exists("stratum1"):
    stratum1_examples = load_ckpt("stratum1")
else:
    print("Building Stratum 1 (Categorical Conflicts)...")
    stratum1_examples = []

    for rule in CATEGORICAL_RULES:
        print(f"  {rule['rule_id']}: {rule['drug_class']} × {rule['conflict_condition']}", end=" → ")

        # Find admissions with matching ICD codes
        all_codes = rule.get("icd10_codes", []) + rule.get("icd9_codes", [])
        # Strip dots for MIMIC format and do prefix match
        code_prefixes = [c.replace(".", "") for c in all_codes]

        mask = diagnoses["_icd_nodot"].str.startswith(tuple(code_prefixes), na=False)
        relevant_admissions = diagnoses.loc[mask, ["subject_id", "hadm_id"]].drop_duplicates()

        if len(relevant_admissions) == 0:
            print("0 admissions, skip")
            continue

        # Find prescriptions of the contraindicated drug for those admissions
        drug_rxs = get_drug_rxs(rule["drug_pattern"], cols=RX_BASE_COLS)

        # Patients who BOTH have the condition AND received the drug
        conflicts = drug_rxs.merge(relevant_admissions, on=["subject_id", "hadm_id"], copy=False)
        conflicts = conflicts.drop_duplicates(subset=["subject_id", "hadm_id"])

        print(f"{len(relevant_admissions):,} admissions, {len(conflicts):,} conflicts")

        # Sample up to quota
        quota = max(1, TARGET_PER_STRATUM // len(CATEGORICAL_RULES))
        if len(conflicts) > quota:
            conflicts = conflicts.sample(n=quota, random_state=SEED)

        for _, row in conflicts.iterrows():
            stratum1_examples.append({
                "example_id": generate_example_id("S1", rule["rule_id"], row["subject_id"], row["hadm_id"]),
                "stratum": 1, "stratum_name": "categorical",
                "conflict_present": True,
                "rule_id": rule["rule_id"],
                "subject_id": int(row["subject_id"]),
                "hadm_id": int(row["hadm_id"]),
                "drug": row["drug"],
                "drug_class": rule["drug_class"],
                "dose_val": row.get("dose_val_rx"),
                "dose_unit": row.get("dose_unit_rx"),
                "route": row.get("route"),
                "conflict_type": rule["conflict_type"],
                "conflict_condition": rule["conflict_condition"],
                "severity": rule["severity"],
                "recommended_action": rule["action"],
                "explanation": rule["explanation"],
                "lab_name": None, "lab_value": None, "lab_unit": None,
                "threshold": None, "operator": None,
            })

    save_ckpt("stratum1", stratum1_examples)

print(f"\n✅ Stratum 1: {len(stratum1_examples)} examples")

In [ ]:
# ── CELL 6: Build Stratum 2 — Numerical-Threshold Conflicts ────────────────

if ckpt_exists("stratum2"):
    stratum2_examples = load_ckpt("stratum2")
else:
    print("Building Stratum 2 (Numerical-Threshold Conflicts)...")
    stratum2_examples = []

    # Pre-filter labs to relevant itemids
    relevant_itemids = list(set(r["lab_itemid"] for r in THRESHOLD_RULES))
    labs_filtered = labevents[labevents["itemid"].isin(relevant_itemids)].copy()
    print(f"  Filtered labs: {len(labs_filtered):,} rows")

    for rule in THRESHOLD_RULES:
        drug_short = rule["drug_pattern"].split("(")[1].split(")")[0].split("|")[0]
        print(f"  {rule['rule_id']}: {drug_short} × {rule['lab_name']} {rule['operator']}{rule['threshold']}", end=" → ")

        drug_rxs = get_drug_rxs(rule["drug_pattern"], cols=RX_BASE_COLS).copy()

        if len(drug_rxs) == 0:
            print("0 rxs, skip")
            continue

        rule_labs = labs_filtered.loc[labs_filtered["itemid"] == rule["lab_itemid"], LAB_BASE_COLS]
        merged = add_nearest_lab_within_window(drug_rxs, rule_labs, start_col="starttime")
        if len(merged) == 0:
            print("0 pairs, skip")
            continue

        # Compute eGFR if needed
        if rule["lab_source"] == "computed":
            merged["lab_final"] = vectorized_egfr(merged["valuenum"], merged["anchor_age"], merged["gender"])
            lab_display, lab_unit_display = "eGFR (CKD-EPI 2021)", "mL/min/1.73m²"
        else:
            merged["lab_final"] = merged["valuenum"]
            lab_display, lab_unit_display = rule["lab_name"], None

        merged = merged.dropna(subset=["lab_final"])

        # Threshold violations
        if rule["operator"] == "<":
            violations = merged[merged["lab_final"] < rule["threshold"]]
        else:
            violations = merged[merged["lab_final"] > rule["threshold"]]

        print(f"{len(merged):,} pairs, {len(violations):,} violations")

        quota = max(1, TARGET_PER_STRATUM // len(THRESHOLD_RULES))
        if len(violations) > quota:
            violations = violations.sample(n=quota, random_state=SEED)

        for _, row in violations.iterrows():
            stratum2_examples.append({
                "example_id": generate_example_id("S2", rule["rule_id"], row["subject_id"], row["hadm_id"]),
                "stratum": 2, "stratum_name": "numerical_threshold",
                "conflict_present": True,
                "rule_id": rule["rule_id"],
                "subject_id": int(row["subject_id"]),
                "hadm_id": int(row["hadm_id"]),
                "drug": row["drug"],
                "drug_class": None,
                "dose_val": row.get("dose_val_rx"),
                "dose_unit": row.get("dose_unit_rx"),
                "route": row.get("route"),
                "conflict_type": "numerical_threshold",
                "conflict_condition": rule["condition"],
                "severity": rule["severity"],
                "recommended_action": rule["action"],
                "explanation": rule["explanation"],
                "lab_name": lab_display,
                "lab_value": float(row["lab_final"]),
                "lab_unit": row.get("valueuom") if lab_unit_display is None else lab_unit_display,
                "threshold": rule["threshold"],
                "operator": rule["operator"],
            })

    save_ckpt("stratum2", stratum2_examples)

print(f"\n✅ Stratum 2: {len(stratum2_examples)} examples")

In [ ]:
# ── CELL 7: Build Stratum 3 — Multi-Hop Causal Conflicts ───────────────────

if ckpt_exists("stratum3"):
    stratum3_examples = load_ckpt("stratum3")
else:
    print("Building Stratum 3 (Multi-Hop Causal Conflicts)...")
    stratum3_examples = []

    relevant_itemids_mh = list(set(r["lab_itemid"] for r in MULTIHOP_RULES))
    labs_filtered_mh = labevents[labevents["itemid"].isin(relevant_itemids_mh)].copy()
    print(f"  Filtered labs: {len(labs_filtered_mh):,} rows")

    for rule in MULTIHOP_RULES:
        print(f"  {rule['rule_id']}: {rule['drug_a_name']} + {rule['drug_b_name']} → {rule['lab_name']} {rule['operator']}{rule['threshold']}", end=" → ")

        # Drug A prescriptions
        drug_a_rxs = get_drug_rxs(rule["drug_a_pattern"], cols=["subject_id","hadm_id","drug","starttime",
                                                   "dose_val_rx","dose_unit_rx","route",
                                                   "gender","anchor_age"])
        drug_a_rxs = drug_a_rxs.rename(columns={"drug":"drug_a","starttime":"starttime_a"})

        # Drug B prescriptions
        drug_b_rxs = get_drug_rxs(rule["drug_b_pattern"], cols=["subject_id","hadm_id","drug","starttime"])
        drug_b_rxs = drug_b_rxs.rename(columns={"drug":"drug_b","starttime":"starttime_b"})

        # Co-prescriptions (same admission)
        coprescribed = drug_a_rxs.merge(drug_b_rxs, on=["subject_id","hadm_id"], how="inner", copy=False)
        coprescribed = coprescribed.drop_duplicates(subset=["subject_id","hadm_id"])

        if len(coprescribed) == 0:
            print("0 co-rx, skip")
            continue

        # Lab values
        rule_labs = labs_filtered_mh.loc[labs_filtered_mh["itemid"] == rule["lab_itemid"], LAB_BASE_COLS]
        merged = add_nearest_lab_within_window(coprescribed, rule_labs, start_col="starttime_a")
        if len(merged) == 0:
            print("0 triples, skip")
            continue

        # Threshold violations
        if rule["operator"] == "<":
            violations = merged[merged["valuenum"] < rule["threshold"]]
        else:
            violations = merged[merged["valuenum"] > rule["threshold"]]

        print(f"{len(coprescribed):,} co-rx, {len(violations):,} violations")

        quota = max(1, TARGET_PER_STRATUM // len(MULTIHOP_RULES))
        if len(violations) > quota:
            violations = violations.sample(n=quota, random_state=SEED)

        for _, row in violations.iterrows():
            stratum3_examples.append({
                "example_id": generate_example_id("S3", rule["rule_id"], row["subject_id"], row["hadm_id"]),
                "stratum": 3, "stratum_name": "multi_hop_causal",
                "conflict_present": True,
                "rule_id": rule["rule_id"],
                "subject_id": int(row["subject_id"]),
                "hadm_id": int(row["hadm_id"]),
                "drug": row["drug_a"],
                "drug_class": None,
                "dose_val": row.get("dose_val_rx"),
                "dose_unit": row.get("dose_unit_rx"),
                "route": row.get("route"),
                "interacting_drug": row["drug_b"],
                "mechanism": rule["mechanism"],
                "conflict_type": "multi_hop_causal",
                "conflict_condition": rule["condition"],
                "severity": rule["severity"],
                "recommended_action": rule["action"],
                "explanation": rule["explanation"],
                "lab_name": rule["lab_name"],
                "lab_value": float(row["valuenum"]),
                "lab_unit": row.get("valueuom"),
                "threshold": rule["threshold"],
                "operator": rule["operator"],
            })

    save_ckpt("stratum3", stratum3_examples)

print(f"\n✅ Stratum 3: {len(stratum3_examples)} examples")

In [ ]:
# ── CELL 8: Build Safe Cases ────────────────────────────────────────────────

if ckpt_exists("safe_cases"):
    safe_examples = load_ckpt("safe_cases")
else:
    print("Building Safe Cases (hard negatives)...")
    safe_examples = []

    # Exclude all conflict admissions
    conflict_pairs = set()
    for ex in stratum1_examples + stratum2_examples + stratum3_examples:
        conflict_pairs.add((ex["subject_id"], ex["hadm_id"]))
    print(f"  Excluding {len(conflict_pairs):,} conflict admissions")

    relevant_itemids = list(set(r["lab_itemid"] for r in THRESHOLD_RULES))
    labs_filtered = labevents[labevents["itemid"].isin(relevant_itemids)].copy()

    # Use first 8 threshold rules for variety
    safe_source_rules = THRESHOLD_RULES[:8]

    for rule in safe_source_rules:
        drug_short = rule["drug_pattern"].split("(")[1].split(")")[0].split("|")[0]
        print(f"  SAFE from {rule['rule_id']} ({drug_short})", end=" → ")

        drug_rxs = get_drug_rxs(rule["drug_pattern"], cols=RX_BASE_COLS).copy()
        rule_labs = labs_filtered[labs_filtered["itemid"] == rule["lab_itemid"]]

        merged = drug_rxs.merge(
            rule_labs[["subject_id","hadm_id","valuenum","valueuom","charttime"]],
            on=["subject_id","hadm_id"], how="inner"
        )
        if len(merged) == 0:
            print("0 pairs, skip")
            continue

        merged["time_diff_h"] = ((merged["charttime"] - merged["starttime"]).dt.total_seconds() / 3600).abs()
        merged = merged[merged["time_diff_h"] <= LAB_WINDOW_HOURS]
        merged = merged.sort_values("time_diff_h").drop_duplicates(subset=["subject_id","hadm_id"], keep="first")

        if rule["lab_source"] == "computed":
            merged["lab_final"] = vectorized_egfr(merged["valuenum"], merged["anchor_age"], merged["gender"])
            lab_display = "eGFR (CKD-EPI 2021)"
        else:
            merged["lab_final"] = merged["valuenum"]
            lab_display = rule["lab_name"]

        merged = merged.dropna(subset=["lab_final"])

        # SAFE: lab clearly within safe range (20% margin from threshold)
        if rule["operator"] == "<":
            safe_cases = merged[merged["lab_final"] >= rule["threshold"] * 1.2]
        else:
            safe_cases = merged[merged["lab_final"] <= rule["threshold"] * 0.8]

        # Remove any that overlap with conflicts
        conflict_index = pd.MultiIndex.from_tuples(conflict_pairs, names=["subject_id", "hadm_id"])
        safe_idx = pd.MultiIndex.from_frame(safe_cases[["subject_id", "hadm_id"]])
        safe_cases = safe_cases.loc[~safe_idx.isin(conflict_index)]

        print(f"{len(safe_cases):,} safe cases")

        quota = max(1, TARGET_SAFE // len(safe_source_rules))
        if len(safe_cases) > quota:
            safe_cases = safe_cases.sample(n=quota, random_state=SEED)

        for _, row in safe_cases.iterrows():
            safe_examples.append({
                "example_id": generate_example_id("SAFE", rule["rule_id"], row["subject_id"], row["hadm_id"]),
                "stratum": 0, "stratum_name": "safe",
                "conflict_present": False,
                "rule_id": f"SAFE-{rule['rule_id']}",
                "subject_id": int(row["subject_id"]),
                "hadm_id": int(row["hadm_id"]),
                "drug": row["drug"],
                "drug_class": None,
                "dose_val": row.get("dose_val_rx"),
                "dose_unit": row.get("dose_unit_rx"),
                "route": row.get("route"),
                "conflict_type": "none",
                "conflict_condition": "none",
                "severity": "none",
                "recommended_action": "safe_to_proceed",
                "explanation": f"Lab ({lab_display} = {float(row['lab_final']):.1f}) within safe range for {row['drug']}.",
                "lab_name": lab_display,
                "lab_value": float(row["lab_final"]),
                "lab_unit": row.get("valueuom"),
                "threshold": rule["threshold"],
                "operator": rule["operator"],
            })

    save_ckpt("safe_cases", safe_examples)

print(f"\n✅ Safe cases: {len(safe_examples)} examples")

In [ ]:
# ── CELL 9: Balance, Split, and Save Final Benchmark ───────────────────────

print("Assembling MedConflict-Bench v2...")

# Cap each stratum
s1 = stratum1_examples[:TARGET_PER_STRATUM]
s2 = stratum2_examples[:TARGET_PER_STRATUM]
s3 = stratum3_examples[:TARGET_PER_STRATUM]
s0 = safe_examples[:TARGET_SAFE]

# Shuffle and combine
all_examples = s1 + s2 + s3 + s0
np.random.seed(SEED)
np.random.shuffle(all_examples)

# Create calibration (20%) / test (80%) splits
n = len(all_examples)
indices = np.random.permutation(n)
cal_size = int(0.2 * n)
cal_set = set(indices[:cal_size].tolist())

for i, ex in enumerate(all_examples):
    ex["split"] = "calibration" if i in cal_set else "test"

# Print summary
print(f"\n{'='*50}")
print(f"MedConflict-Bench v2 — Final Summary")
print(f"{'='*50}")
for label, data in [("Safe (S0)", s0), ("Categorical (S1)", s1),
                     ("Numerical (S2)", s2), ("Multi-hop (S3)", s3)]:
    print(f"  {label:25s}: {len(data):>5}")
print(f"  {'TOTAL':25s}: {len(all_examples):>5}")
print(f"  {'Calibration split':25s}: {sum(1 for e in all_examples if e['split']=='calibration'):>5}")
print(f"  {'Test split':25s}: {sum(1 for e in all_examples if e['split']=='test'):>5}")
print(f"  {'Unique subjects':25s}: {len(set(e['subject_id'] for e in all_examples)):>5}")
print(f"  {'Unique admissions':25s}: {len(set(e['hadm_id'] for e in all_examples)):>5}")

# ---- Save JSONL ----
jsonl_path = OUTPUT_DIR / "medconflict_bench_v2.jsonl"
with open(jsonl_path, "w") as f:
    for ex in all_examples:
        f.write(json.dumps(ex, default=str) + "\n")
print(f"\n💾 Saved: {jsonl_path}")

# ---- Save CSV ----
csv_path = OUTPUT_DIR / "medconflict_bench_v2.csv"
pd.DataFrame(all_examples).to_csv(csv_path, index=False)
print(f"💾 Saved: {csv_path}")

# ---- Save splits metadata ----
splits_path = OUTPUT_DIR / "splits_v2.json"
meta = {
    "version": "2.0",
    "created": pd.Timestamp.now().isoformat(),
    "seed": SEED,
    "lab_window_hours": LAB_WINDOW_HOURS,
    "total_examples": len(all_examples),
    "strata": {
        "0_safe": len(s0),
        "1_categorical": len(s1),
        "2_numerical_threshold": len(s2),
        "3_multi_hop_causal": len(s3),
    },
    "splits": {
        "calibration": sum(1 for e in all_examples if e["split"] == "calibration"),
        "test": sum(1 for e in all_examples if e["split"] == "test"),
    },
    "rules_count": {
        "categorical": len(CATEGORICAL_RULES),
        "threshold": len(THRESHOLD_RULES),
        "multihop": len(MULTIHOP_RULES),
    },
    "mimic_iv_version": "3.1",
    "egfr_formula": "CKD-EPI 2021 (race-free)",
}
with open(splits_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"💾 Saved: {splits_path}")

print(f"\n✅ MedConflict-Bench v2 BUILD COMPLETE")
print(f"   All files in: {OUTPUT_DIR}")

In [ ]:
# ── CELL 10: Quick Validation & Sanity Checks ──────────────────────────────

print("Running sanity checks...\n")

df = pd.read_csv(OUTPUT_DIR / "medconflict_bench_v2.csv")

# 1. No duplicate example IDs
assert df["example_id"].nunique() == len(df), "❌ Duplicate example IDs found!"
print("✅ All example IDs unique")

# 2. All strata present
for s in [0, 1, 2, 3]:
    count = (df["stratum"] == s).sum()
    assert count > 0, f"❌ Stratum {s} has 0 examples!"
    print(f"✅ Stratum {s}: {count} examples")

# 3. Conflict labels match strata
assert (df[df["stratum"] == 0]["conflict_present"] == False).all(), "❌ Safe cases marked as conflicts!"
assert (df[df["stratum"] > 0]["conflict_present"] == True).all(), "❌ Conflict cases marked as safe!"
print("✅ Labels consistent with strata")

# 4. No subject appears in both conflict and safe sets
conflict_subjects = set(df[df["stratum"] > 0]["hadm_id"])
safe_subjects = set(df[df["stratum"] == 0]["hadm_id"])
overlap = conflict_subjects & safe_subjects
if overlap:
    print(f"⚠️  {len(overlap)} admission(s) in both conflict and safe sets (check manually)")
else:
    print("✅ No admission overlap between conflict and safe sets")

# 5. Numerical values present where expected
s2_df = df[df["stratum"] == 2]
assert s2_df["lab_value"].notna().all(), "❌ Missing lab values in Stratum 2!"
assert s2_df["threshold"].notna().all(), "❌ Missing thresholds in Stratum 2!"
print("✅ All Stratum 2 examples have lab values and thresholds")

# 6. Multi-hop examples have interacting drug
s3_df = df[df["stratum"] == 3]
if "interacting_drug" in s3_df.columns:
    assert s3_df["interacting_drug"].notna().all(), "❌ Missing interacting drug in Stratum 3!"
    print("✅ All Stratum 3 examples have interacting drug")

# 7. Splits present
assert set(df["split"].unique()) == {"calibration", "test"}, "❌ Missing split assignments!"
print("✅ Calibration/test splits assigned")

# Per-rule distribution
print("\n--- Examples per rule ---")
for s_name, s_val in [("S1-Categorical", 1), ("S2-Numerical", 2), ("S3-Multi-hop", 3)]:
    subset = df[df["stratum"] == s_val]
    print(f"\n{s_name}:")
    for rule_id, count in subset["rule_id"].value_counts().items():
        print(f"  {rule_id}: {count}")

print("\n✅ All sanity checks passed!")

In [ ]:
#!/usr/bin/env python3
"""
05_calibrate_v3.py — Conformal Calibration on REAL KG C-Scores
================================================================

Uses medconflict_bench_v2_scored.jsonl (output of 04b_cscore_benchmark.py)
which has REAL C-scores from Neo4j KG lookups.

This is the correct approach:
  - Ground truth: conflict_present (from MIMIC-IV annotations)
  - Prediction: kg_cscore (from Neo4j KG lookup — may miss some conflicts)
  - Conformal calibration finds τ so that missed conflict rate ≤ α

Run on Colab as Cell 11 (after copying scored file from lab machine to Drive).
"""

import json
import numpy as np
from pathlib import Path

# ============================================================================
# CONFIG
# ============================================================================

BENCH_DIR = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench")
# Use the SCORED file from lab machine:
SCORED_FILE = BENCH_DIR / "medconflict_bench_v2_scored.jsonl"
# Fallback to original if scored doesn't exist:
FALLBACK_FILE = BENCH_DIR / "medconflict_bench_v2.jsonl"

OUTPUT_DIR = BENCH_DIR / "calibration"
ALPHA_VALUES = [0.01, 0.05, 0.10, 0.20]
SEED = 42
np.random.seed(SEED)


def evaluate(scores, labels, tau):
    """C >= τ → ABSTAIN, C < τ → predict safe."""
    n = len(scores)
    abstained = scores >= tau
    predicted_safe = ~abstained

    coverage = predicted_safe.sum() / n

    safe_pred_labels = labels[predicted_safe]
    if len(safe_pred_labels) > 0:
        accuracy = (safe_pred_labels == 0).sum() / len(safe_pred_labels)
        error_rate = 1 - accuracy
    else:
        accuracy, error_rate = 1.0, 0.0

    conflicts = labels == 1
    recall = (abstained & conflicts).sum() / conflicts.sum() if conflicts.sum() > 0 else 1.0
    missed = (predicted_safe & conflicts).sum()

    precision = (abstained & conflicts).sum() / abstained.sum() if abstained.sum() > 0 else 1.0

    return {
        "tau": round(float(tau), 4),
        "coverage": round(float(coverage), 4),
        "accuracy": round(float(accuracy), 4),
        "error_rate": round(float(error_rate), 4),
        "recall": round(float(recall), 4),
        "precision": round(float(precision), 4),
        "n_abstained": int(abstained.sum()),
        "n_safe_pred": int(predicted_safe.sum()),
        "n_missed": int(missed),
        "n_total": n,
    }


def calibrate(conflict_scores, alpha):
    """Find τ: only α fraction of conflicts have score < τ."""
    n = len(conflict_scores)
    if n == 0:
        return 0.5
    sorted_s = np.sort(conflict_scores)
    idx = int(np.ceil(alpha * (n + 1))) - 1
    idx = max(0, min(idx, n - 1))
    return sorted_s[idx]


def main():
    print("=" * 70)
    print("Conformal Calibration v3 — Using Real KG C-Scores")
    print("=" * 70)

    # Load scored benchmark
    if SCORED_FILE.exists():
        bench_path = SCORED_FILE
        score_field = "kg_cscore"
        print(f"\nUsing REAL KG scores: {bench_path}")
    else:
        print(f"\n⚠ Scored file not found: {SCORED_FILE}")
        print(f"  Using fallback (ground-truth C-scores): {FALLBACK_FILE}")
        bench_path = FALLBACK_FILE
        score_field = None

    examples = [json.loads(line) for line in open(bench_path)]
    cal_ex = [e for e in examples if e.get("split") == "calibration"]
    test_ex = [e for e in examples if e.get("split") == "test"]
    print(f"  Total: {len(examples)} | Cal: {len(cal_ex)} | Test: {len(test_ex)}")

    # Extract scores
    if score_field and score_field in cal_ex[0]:
        cal_scores = np.array([e[score_field] for e in cal_ex])
        test_scores = np.array([e[score_field] for e in test_ex])
        print(f"  Using field: {score_field}")
    else:
        # Fallback: compute from ground truth (less useful but works)
        print(f"  ⚠ Computing from ground truth (run 04b on lab machine for better results)")
        w1, w2 = 0.7, 0.3
        def gt_score(e):
            if not e.get("conflict_present"):
                return 0.0
            pv = 1.0
            nd = 0.0
            lv = e.get("lab_value")
            thr = e.get("threshold")
            op = e.get("operator") or e.get("op")
            if lv is not None and thr is not None and thr != 0:
                if op == "<":
                    nd = max(0, (thr - lv) / thr)
                elif op == ">":
                    nd = max(0, (lv - thr) / thr)
                nd = min(nd, 1.0)
            return w1 * pv + w2 * nd
        cal_scores = np.array([gt_score(e) for e in cal_ex])
        test_scores = np.array([gt_score(e) for e in test_ex])

    cal_labels = np.array([1 if e["conflict_present"] else 0 for e in cal_ex])
    test_labels = np.array([1 if e["conflict_present"] else 0 for e in test_ex])

    # Score distribution
    print(f"\n  Score distribution:")
    for s in [0, 1, 2, 3]:
        mask = np.array([e.get("stratum") == s for e in test_ex])
        if mask.sum() == 0: continue
        ss = test_scores[mask]
        name = {0: "Safe", 1: "Categorical", 2: "Numerical", 3: "Multi-hop"}[s]
        print(f"    S{s} ({name}): n={mask.sum()}, "
              f"min={ss.min():.4f}, mean={ss.mean():.4f}, max={ss.max():.4f}, "
              f"nonzero={np.count_nonzero(ss)}")

    # Calibrate
    print(f"\n  {'α':<8} {'τ':<10} {'Coverage':<10} {'Accuracy':<10} {'Recall':<10} {'Missed':<8} {'Error%':<8}")
    print("  " + "-" * 64)

    results = {}
    for alpha in ALPHA_VALUES:
        tau = calibrate(cal_scores[cal_labels == 1], alpha)
        m = evaluate(test_scores, test_labels, tau)

        results[str(alpha)] = {"alpha": alpha, "tau": m["tau"], "test": m,
                                "cal": evaluate(cal_scores, cal_labels, tau)}

        print(f"  {alpha:<8} {tau:<10.4f} {m['coverage']:<10.4f} {m['accuracy']:<10.4f} "
              f"{m['recall']:<10.4f} {m['n_missed']:<8} {m['error_rate']:<8.4f}")

    # Per-stratum at α=0.05
    print(f"\n  Per-stratum (α=0.05, τ={results['0.05']['tau']}):")
    tau05 = results["0.05"]["tau"]
    for s in [0, 1, 2, 3]:
        mask = np.array([e.get("stratum") == s for e in test_ex])
        if mask.sum() == 0: continue
        m = evaluate(test_scores[mask], test_labels[mask], tau05)
        name = {0: "Safe", 1: "Categorical", 2: "Numerical", 3: "Multi-hop"}[s]
        print(f"    S{s} ({name}): recall={m['recall']:.3f}, coverage={m['coverage']:.3f}, missed={m['n_missed']}")

    # Save
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    out = {
        "method": "split_conformal_v3_real_kg_scores",
        "score_source": score_field or "ground_truth_fallback",
        "alphas": results,
        "n_cal": len(cal_ex),
        "n_test": len(test_ex),
    }
    with open(OUTPUT_DIR / "calibration_results_v3.json", "w") as f:
        json.dump(out, f, indent=2, default=str)

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Distribution
        ax = axes[0]
        bins = np.linspace(0, 1, 40)
        ax.hist(test_scores[test_labels == 0], bins=bins, alpha=0.6, color="green",
                label=f"Safe (n={int((test_labels==0).sum())})", density=True)
        ax.hist(test_scores[test_labels == 1], bins=bins, alpha=0.6, color="red",
                label=f"Conflict (n={int((test_labels==1).sum())})", density=True)
        ax.axvline(x=tau05, color="black", ls="--", lw=2, label=f"τ(α=0.05)={tau05:.3f}")
        ax.set_xlabel("C-Score"); ax.set_ylabel("Density")
        ax.set_title("C-Score Distribution (KG)"); ax.legend(); ax.grid(True, alpha=0.3)

        # Coverage-accuracy
        ax = axes[1]
        taus = np.linspace(0, 1, 200)
        covs = [evaluate(test_scores, test_labels, t)["coverage"] for t in taus]
        accs = [evaluate(test_scores, test_labels, t)["accuracy"] for t in taus]
        ax.plot(covs, accs, "b-", lw=2)
        ax.set_xlabel("Coverage"); ax.set_ylabel("Accuracy")
        ax.set_title("Coverage–Accuracy Curve"); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "calibration_plots_v3.png", dpi=150)
        plt.close()
        print(f"\n  Plot saved: {OUTPUT_DIR / 'calibration_plots_v3.png'}")
    except Exception as e:
        print(f"  Plot error: {e}")

    print(f"\n{'='*70}")
    print(f"DONE — Results in {OUTPUT_DIR}")
    print(f"{'='*70}")
    r = results["0.05"]
    print(f"  Recommended (α=0.05): τ={r['tau']}")
    print(f"  Coverage: {r['test']['coverage']:.1%} | Recall: {r['test']['recall']:.1%} | Missed: {r['test']['n_missed']}")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
"""
Diagnostic: Why are Safe cases scoring HIGH and Categorical scoring LOW?
Run this on Colab as a new cell to inspect the scored data.
"""

import json
from pathlib import Path
from collections import Counter

SCORED = Path("/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/medconflict_bench_v2_scored.jsonl")
examples = [json.loads(line) for line in open(SCORED)]

print("=" * 70)
print("DIAGNOSTIC: Inspecting KG C-scores")
print("=" * 70)

# --- S0 (Safe) with HIGH scores ---
print("\n--- SAFE cases with HIGH kg_cscore (should be 0!) ---")
safe_high = [e for e in examples if e["stratum"] == 0 and e["kg_cscore"] > 0.5]
print(f"Count: {len(safe_high)} / {sum(1 for e in examples if e['stratum']==0)}")
for e in safe_high[:10]:
    print(f"  Drug: {e['drug']:<30s} | lab: {e.get('lab_name','?')}={e.get('lab_value','?')} | "
          f"kg_cscore: {e['kg_cscore']} | violations: {e['kg_trace']['n_violations']} | "
          f"trace: {e['kg_trace'].get('violations_summary', [])}")

# --- S1 (Categorical) with LOW scores ---
print("\n--- CATEGORICAL conflicts with LOW kg_cscore (should be >0!) ---")
cat_low = [e for e in examples if e["stratum"] == 1 and e["kg_cscore"] == 0]
print(f"Count: {len(cat_low)} / {sum(1 for e in examples if e['stratum']==1)}")
for e in cat_low[:10]:
    print(f"  Drug: {e['drug']:<30s} | condition: {e.get('conflict_condition','?')} | "
          f"rule: {e['rule_id']} | kg_cscore: {e['kg_cscore']} | "
          f"paths_checked: {e['kg_trace']['paths_checked']} | paths_violated: {e['kg_trace']['paths_violated']}")

# --- S1 with HIGH scores (the 35 that worked) ---
print("\n--- CATEGORICAL conflicts WITH detection (kg_cscore > 0) ---")
cat_high = [e for e in examples if e["stratum"] == 1 and e["kg_cscore"] > 0]
print(f"Count: {len(cat_high)} / {sum(1 for e in examples if e['stratum']==1)}")
for e in cat_high[:10]:
    print(f"  Drug: {e['drug']:<30s} | condition: {e.get('conflict_condition','?')} | "
          f"rule: {e['rule_id']} | kg_cscore: {e['kg_cscore']}")

# --- Drug names in each stratum ---
print("\n--- Top drugs in Safe cases (with high scores) ---")
safe_drugs = [e["drug"].split()[0].lower() for e in safe_high]
for drug, cnt in Counter(safe_drugs).most_common(10):
    print(f"  {drug}: {cnt}")

print("\n--- Top drugs in Categorical (missed) ---")
cat_drugs = [e["drug"].split()[0].lower() for e in cat_low]
for drug, cnt in Counter(cat_drugs).most_common(10):
    print(f"  {drug}: {cnt}")

# --- Check what KG finds for a safe drug ---
print("\n--- Sample SAFE case trace ---")
if safe_high:
    e = safe_high[0]
    print(f"  Drug: {e['drug']}")
    print(f"  Lab: {e.get('lab_name')}={e.get('lab_value')}")
    print(f"  Threshold: {e.get('operator')}{e.get('threshold')}")
    print(f"  Trace: {json.dumps(e['kg_trace'], indent=2)}")